In [1]:
import sklearn, xgboost, shap, optuna, pandas, numpy, platform
print(f"Python:      {platform.python_version()}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost:     {xgboost.__version__}")
print(f"shap:        {shap.__version__}")
print(f"optuna:      {optuna.__version__}")
print(f"pandas:      {pandas.__version__}")
print(f"numpy:       {numpy.__version__}")

C:\Users\miy\miniconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python:      3.10.15
scikit-learn: 1.5.2
xgboost:     1.7.5
shap:        0.46.0
optuna:      4.5.0
pandas:      2.3.3
numpy:       1.26.4


In [3]:
# CPU 정보
import platform
print(platform.processor())

# RAM
import psutil
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")

Intel64 Family 6 Model 191 Stepping 2, GenuineIntel
RAM: 127.7 GB


# DenSHAP Demo Notebook

**DenSHAP: Density-Aware Background Dataset Construction for Counterfactual Shapley Explanations**

This notebook reproduces the main experimental results on three benchmark datasets:
- **HELOC** (9,871 samples, 23 features)
- **Wine Quality** (6,497 samples, 11 features)
- **LendingClub** (sampled 20,000, 20 features)

**Methods compared:**
| Method | Background dataset |
|--------|-------------------|
| SHAP_TRAIN | Random from all training data |
| SHAP_D_LAB | Random from opposite-label data |
| SHAP_D_PRED | Random from opposite-predicted data |
| CF_SHAP | IQR-normalized KNN (distance only) |
| **DenSHAP** | **LOF-weighted KNN (distance + density)** |

> **Prerequisites:** Place `heloc.csv`, `lendingclub.csv`, `wine.csv` in `data/` folder.


In [2]:
%run run_experiment.py


#################################################################
  k_lof sweep: 10  →  saving to results\klof10/
#################################################################

  Dataset : HELOC
  k_lof   : 10
  output  : results\klof10/
[HELOC] loading...
  Number of columns: 24
  Features used (23 features): ['AverageMInFile', 'ExternalRiskEstimate', 'MSinceMostRecentDelq', 'MSinceMostRecentInqexcl7days', 'MSinceMostRecentTradeOpen'] ...
  Total samples: 9871
  Train: (7896, 23) | Test: (1975, 23)
  Label distribution (train): {0: 3788, 1: 4108}
  Minority class ratio: 0.520

[Model] Running Optuna tuning (30 trials) ...
  Optuna done | Best CV AUC: 0.8098
  Best params: {'n_estimators': 427, 'max_depth': 3, 'learning_rate': 0.015928850423477626, 'subsample': 0.7727431143998708, 'colsample_bytree': 0.8209823503143753, 'min_child_weight': 9, 'gamma': 0.8612290014449612, 'reg_alpha': 0.012324531280088652, 'reg_lambda': 0.05970576865103405, 'eval_metric': 'logloss', 'random_state':

In [3]:
import pandas as pd
import os

k_lof_list = [10, 20, 30, 50]
datasets   = ['heloc', 'wine', 'lendingclub']

rows = []
for k in k_lof_list:
    folder = f'results/klof{k}'
    for ds in datasets:
        path = f'{folder}/{ds}_bds_by_group.csv'
        if not os.path.exists(path):
            print(f"Missing: {path}")
            continue
        df   = pd.read_csv(path)
        hard = df[df['Group'] == 'Hard'].iloc[0]
        rows.append({
            'k_lof':        k,
            'Dataset':      ds.upper(),
            'N':            int(hard['N']),
            'CF_SHAP_BDS':  hard['CF_SHAP_BDS'],
            'DenSHAP_BDS':  hard['DenSHAP_BDS'],
            'Delta_pct':    hard['Delta_pct'],
        })

result = pd.DataFrame(rows)
print(result.to_string(index=False))

 k_lof     Dataset    N  CF_SHAP_BDS  DenSHAP_BDS  Delta_pct
    10       HELOC  734       0.9412       0.9654     2.5631
    10        WINE  358       0.9660       0.9774     1.1826
    10 LENDINGCLUB 1686       0.9690       0.9760     0.7242
    20       HELOC  708       0.9385       0.9641     2.7216
    20        WINE  321       0.9632       0.9766     1.3913
    20 LENDINGCLUB 1595       0.9651       0.9755     1.0765
    30       HELOC  683       0.9372       0.9663     3.0959
    30        WINE  289       0.9617       0.9763     1.5206
    30 LENDINGCLUB 1509       0.9603       0.9737     1.3882
    50       HELOC  692       0.9366       0.9668     3.2330
    50        WINE  284       0.9574       0.9754     1.8872
    50 LENDINGCLUB 1476       0.9521       0.9709     1.9798


In [7]:
import pandas as pd
from scipy.stats import wilcoxon

datasets = ['heloc', 'wine', 'lendingclub']

for ds in datasets:
    cf  = pd.read_csv(f'results/klof20/{ds}_cfshap.csv')
    ds_ = pd.read_csv(f'results/klof20/{ds}_denshap.csv')
    
    for group in ['Easy', 'Medium', 'Hard']:
        mask    = ds_['difficulty_group'] == group
        df_g    = ds_[mask]
        if len(df_g) == 0:
            continue
        ds_bds  = df_g['DenSHAP_bds'].values
        cf_bds  = cf.loc[df_g.index, 'CF_SHAP_bds'].values
        diff    = ds_bds - cf_bds
        if (diff == 0).all():
            print(f"{ds.upper()} {group}: all zeros — skip")
            continue
        try:
            stat, p = wilcoxon(ds_bds, cf_bds, alternative='greater')
            print(f"{ds.upper():12s} {group:6s} N={len(df_g):4d}  "
                  f"CF={cf_bds.mean():.4f}  DS={ds_bds.mean():.4f}  "
                  f"p={p:.2e}")
        except Exception as e:
            print(f"{ds.upper()} {group}: {e}")

HELOC        Easy   N= 928  CF=0.9851  DS=0.9777  p=1.00e+00
HELOC        Medium N= 339  CF=0.9636  DS=0.9728  p=2.49e-08
HELOC        Hard   N= 708  CF=0.9385  DS=0.9641  p=4.19e-52
WINE         Easy   N= 391  CF=0.9778  DS=0.9808  p=1.40e-01
WINE         Medium N= 134  CF=0.9762  DS=0.9797  p=1.88e-01
WINE         Hard   N= 321  CF=nan  DS=nan  p=nan
LENDINGCLUB  Easy   N=1682  CF=0.9803  DS=0.9786  p=1.00e+00
LENDINGCLUB  Medium N= 723  CF=0.9779  DS=0.9782  p=3.34e-01
LENDINGCLUB  Hard   N=1595  CF=0.9651  DS=0.9755  p=2.13e-33


In [5]:
import pandas as pd

cf  = pd.read_csv('results/klof20/wine_cfshap.csv')
ds_ = pd.read_csv('results/klof20/wine_denshap.csv')

mask = ds_['difficulty_group'] == 'Hard'
df_g = ds_[mask]

print("DenSHAP Hard N:", len(df_g))
print("CF_SHAP_bds column exists:", 'CF_SHAP_bds' in cf.columns)
print("CF_SHAP_bds sample:", cf.loc[df_g.index, 'CF_SHAP_bds'].head(10).values)
print("DenSHAP_bds sample:", df_g['DenSHAP_bds'].head(10).values)

DenSHAP Hard N: 321
CF_SHAP_bds column exists: True
CF_SHAP_bds sample: [0.99115227 0.97196048 0.99108984 0.97254824 0.98455147 0.99354563
 0.96436483        nan 0.98083203 0.98762453]
DenSHAP_bds sample: [0.99405329 0.98113324 0.98981586 0.97177251 0.95691021 0.95995968
 0.9426388         nan 0.99799146 0.99430185]


In [8]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

cf  = pd.read_csv('results/klof20/wine_cfshap.csv')
ds_ = pd.read_csv('results/klof20/wine_denshap.csv')

mask = ds_['difficulty_group'] == 'Hard'
df_g = ds_[mask]

ds_bds = df_g['DenSHAP_bds'].values
cf_bds = cf.loc[df_g.index, 'CF_SHAP_bds'].values

# nan 제외
valid = ~(np.isnan(ds_bds) | np.isnan(cf_bds))
print(f"Total: {len(ds_bds)}, Valid: {valid.sum()}, NaN removed: {(~valid).sum()}")

ds_valid = ds_bds[valid]
cf_valid = cf_bds[valid]

print(f"CF_SHAP_BDS mean: {cf_valid.mean():.4f}")
print(f"DenSHAP_BDS mean: {ds_valid.mean():.4f}")
print(f"Delta: {(ds_valid.mean()-cf_valid.mean())/cf_valid.mean()*100:.4f}%")

stat, p = wilcoxon(ds_valid, cf_valid, alternative='greater')
print(f"p-value: {p:.2e}")

Total: 321, Valid: 318, NaN removed: 3
CF_SHAP_BDS mean: 0.9633
DenSHAP_BDS mean: 0.9767
Delta: 1.3914%
p-value: 4.74e-07
